In [1]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from shapely.geometry import box
import os

In [2]:
def mask_raster_with_shapefile(raster_path, shapefile_path, output_dir):
    # Read the raster dataset
    with rasterio.open(raster_path) as src:
        # Read the shapefile
        gdf = gpd.read_file(shapefile_path)

        for idx, feature in gdf.iterrows():
            # Get the geometry of the feature
            geo_feature = gpd.GeoSeries(feature['geometry'])

            # Crop the raster to the bounding box of the feature
            out_image, out_transform = mask(src, geo_feature, crop=True)

            # Update metadata for the cropped raster
            out_meta = src.meta.copy()
            out_meta.update({
                'height': out_image.shape[1],
                'width': out_image.shape[2],
                'transform': out_transform,
                'compress': 'deflate',  # Set compression to deflate
                'tiled': True,  # Enable tiling
            })

            # Construct the output file name using the base name of the input raster
            base_name = os.path.splitext(os.path.basename(raster_path))[0]
            output_filename = f"{output_dir}/{base_name}_split_{idx + 1}.tif"

            # Write the cropped raster to a new file
            with rasterio.open(output_filename, 'w', **out_meta) as dest:
                dest.write(out_image)

            print(f"Output raster created: {output_filename}")

In [4]:
if __name__ == "__main__":
    # Replace these paths with your raster and shapefile paths
    raster_path = r"C:\Users\admin\Downloads\copernicus_tree_cover_density_HR\TCD_2018_010m_eu_04326_V2_0_reclass_mask.tif"
    shapefile_path = r"C:\Users\admin\Downloads\copernicus_tree_cover_density_HR\focus_countries_dissolved_split.shp"

    # Replace this path with the directory where you want to save the output rasters
    output_dir = r"C:\Users\admin\Downloads\copernicus_tree_cover_density_HR"

    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    mask_raster_with_shapefile(raster_path, shapefile_path, output_dir)


ValueError: Input shapes do not overlap raster.